# EchoFactory - FAN: Training v3 (STgram-MFN Optimized)

## Perbaikan dari fanechofac3:
| | fanechofac3 | **fanechofac7 (baru)** |
|---|---|---|
| Fitur Branch 1 | MFCC+Delta (lemah) | **T-gram STFT (kuat)** |
| embed_dim | 256 | **128** (4 kelas tidak butuh 256) |
| ArcFace s,m | s=12, m=0.3 | **s=30, m=0.5** (margin lebih kuat) |
| Epochs | 100 | **150** |
| SpecAugment | ❌ | **✅** (time + freq masking) |
| LR warmup | 10 ep | **20 ep** (lebih stabil) |


In [ ]:
import os, gc, json, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# =============================================
# KONFIGURASI
# =============================================
MACHINE_TYPE = 'fan'

# Fitur dari fanechofac6
FEAT_DIR  = '/kaggle/input/notebooks/muhammadmuhibin/fanechofac6/features_v3'

BATCH_SIZE   = 64
EPOCHS       = 150
LR           = 5e-4      # LR peak setelah warmup
WARMUP_EP    = 20        # Lebih panjang warmup untuk stabilitas
WEIGHT_DECAY = 1e-3
EMBED_DIM    = 128       # 4 kelas → 128 cukup, lebih compact
ARC_S        = 30.0      # ArcFace scale yang lebih besar
ARC_M        = 0.5       # ArcFace margin standar
OUT_MODEL    = f'/kaggle/working/stgram_mfn_{MACHINE_TYPE}_v3.pt'

print(f'Machine: {MACHINE_TYPE.upper()}')
print(f'embed_dim={EMBED_DIM} | ArcFace s={ARC_S}, m={ARC_M}')
print(f'Epochs={EPOCHS} | Warmup={WARMUP_EP} | LR_peak={LR}')
print(f'SpecAugment: AKTIF')


In [ ]:
# =============================================
# SPECAUGMENT — Augmentasi untuk meningkatkan generalisasi
# =============================================
class SpecAugment(nn.Module):
    """
    SpecAugment: masking secara acak di dimensi frekuensi dan waktu.
    Mendorong model belajar representasi yang lebih robust.
    Hanya aktif saat training, tidak saat inference.
    """
    def __init__(self, freq_mask=15, time_mask=20, num_freq_masks=2, num_time_masks=2):
        super().__init__()
        self.fm = freq_mask
        self.tm = time_mask
        self.nfm = num_freq_masks
        self.ntm = num_time_masks

    def forward(self, x):
        """x: (B, 1, H, W) = (batch, channel, freq, time)"""
        if not self.training: return x
        B, C, H, W = x.shape
        x = x.clone()
        for _ in range(self.nfm):
            f = torch.randint(0, self.fm, (1,)).item()
            f0 = torch.randint(0, max(1, H - f), (1,)).item()
            x[:, :, f0:f0+f, :] = 0.0
        for _ in range(self.ntm):
            t = torch.randint(0, self.tm, (1,)).item()
            t0 = torch.randint(0, max(1, W - t), (1,)).item()
            x[:, :, :, t0:t0+t] = 0.0
        return x

print('SpecAugment defined')


In [ ]:
# =============================================
# MODEL ARCHITECTURE
# =============================================
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False),
            nn.BatchNorm2d(oc), nn.PReLU(oc)
        )
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNPReLU(ic, ic, s=s, g=ic),
            ConvBNPReLU(ic, oc, k=1, p=0)
        )
    def forward(self, x): return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.aug = SpecAugment(freq_mask=15, time_mask=20)
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2), DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2), DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2), DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2), nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed)
        )
    def forward(self, x):
        x = self.aug(x)   # SpecAugment hanya aktif saat training
        return self.head(self.enc(x))

class ArcFace(nn.Module):
    def __init__(self, ed, nc, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m);  self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m
    def forward(self, feat, labels):
        cos = F.normalize(feat) @ F.normalize(self.W).T
        sin = (1.0 - cos**2 + 1e-8).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = F.one_hot(labels, cos.shape[1]).float()
        out = (one_hot * phi + (1.0 - one_hot) * cos) * self.s
        return F.cross_entropy(out, labels)

class STgramMFN(nn.Module):
    def __init__(self, nc, ed=128):
        super().__init__()
        self.mel   = MobileFaceNet(ed)
        self.tgram = MobileFaceNet(ed)
        self.fuse  = nn.Sequential(
            nn.Linear(ed * 2, ed),
            nn.BatchNorm1d(ed),
            nn.PReLU(ed)
        )
        self.arc = ArcFace(ed, nc, ARC_S, ARC_M)

    def forward(self, mel, tg, labels=None):
        feat = F.normalize(
            self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1)),
            dim=1
        )
        if labels is not None:
            return feat, self.arc(feat, labels)
        return feat

print('STgramMFN v3 defined (embed_dim=128, ArcFace s=30, m=0.5, SpecAugment aktif)')


In [ ]:
# =============================================
# DATASET & DATALOADER
# =============================================
class MIMIIDataset(Dataset):
    def __init__(self, machine, feat_dir, cond='normal'):
        path = os.path.join(feat_dir, f'{machine}_{cond}.pt')
        data = torch.load(path, map_location='cpu')
        self.feats  = data['features']
        self.labels = data['labels']
        self.n_cls  = int(self.labels.max().item()) + 1
    def __len__(self): return len(self.feats)
    def __getitem__(self, idx):
        f = self.feats[idx]
        return f[0:1], f[1:2], self.labels[idx]

train_ds = MIMIIDataset(MACHINE_TYPE, FEAT_DIR, 'normal')
N_CLASSES = train_ds.n_cls
train_dl  = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=2, pin_memory=True)
print(f'DataLoader: {len(train_dl)} batches/epoch | {N_CLASSES} machine IDs')
print(f'Total training samples: {len(train_ds)}')


In [ ]:
# =============================================
# TRAINING LOOP dengan LR Warmup + Cosine Decay
# =============================================
model     = STgramMFN(N_CLASSES, EMBED_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler('cuda') if torch.cuda.is_available() else GradScaler('cpu')

# LR Scheduler: Linear Warmup → Cosine Annealing
def get_lr(epoch):
    if epoch <= WARMUP_EP:
        return LR * epoch / WARMUP_EP
    progress = (epoch - WARMUP_EP) / (EPOCHS - WARMUP_EP)
    return LR * 0.5 * (1 + math.cos(math.pi * progress))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'STgram-MFN v3 on {device} | {n_params:,} params ({n_params/1e6:.2f}M)')

best_loss = float('inf')
losses    = []
t0        = time.time()

print(f'Training {MACHINE_TYPE.upper()} v3 | {EPOCHS} epochs | Warmup {WARMUP_EP} ep')
print('-' * 65)

for ep in range(1, EPOCHS + 1):
    # Update LR
    lr_now = get_lr(ep)
    for pg in optimizer.param_groups: pg['lr'] = lr_now

    model.train()
    epoch_loss = 0.0

    for mel, tg, lab in train_dl:
        mel, tg, lab = mel.to(device), tg.to(device), lab.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            _, loss = model(mel, tg, lab)
        scaler.scale(loss).backward()
        # Gradient clipping untuk stabilitas
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    avg = epoch_loss / len(train_dl)
    losses.append(avg)

    if avg < best_loss:
        best_loss = avg
        torch.save({
            'epoch': ep, 'model_state': model.state_dict(),
            'best_loss': best_loss, 'machine': MACHINE_TYPE,
            'n_classes': N_CLASSES, 'embed_dim': EMBED_DIM,
            'arc_s': ARC_S, 'arc_m': ARC_M
        }, OUT_MODEL)
        tag = ' <- BEST'
    else:
        tag = ''

    if ep % 10 == 0 or ep == 1 or ep == EPOCHS:
        elapsed = (time.time() - t0) / 60
        print(f'Ep {ep:3d}/{EPOCHS} | Loss: {avg:.4f} | LR: {lr_now:.2e} | {elapsed:.1f}min{tag}')

print(f'\nTraining selesai! Best loss: {best_loss:.4f} → {OUT_MODEL}')
if torch.cuda.is_available():
    print(f'Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')


In [ ]:
# === PLOT LOSS CURVE ===
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', lw=1.5, label='Training Loss')
plt.axvline(x=WARMUP_EP-1, color='orange', linestyle='--', alpha=0.7, label=f'End Warmup (ep {WARMUP_EP})')
plt.axhline(y=best_loss, color='green', linestyle='--', alpha=0.7, label=f'Best Loss: {best_loss:.4f}')
plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('Loss (log scale)')
plt.title(f'Training Loss Curve — FAN v3 (embed={EMBED_DIM}, s={ARC_S}, m={ARC_M})', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curve_fan_v3.png', dpi=150)
plt.show()
print('Loss curve saved.')
print('\nfanechofac7 SELESAI! → Lanjut ke fanechofac8 (Evaluasi v3)')
